# <font color="#418FDE" size="6.5" uppercase>**Filter Wrapper Methods**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply filter methods to remove low-variance or statistically weak features. 
- Use univariate feature selection with scoring functions appropriate to classification or regression. 
- Perform recursive feature elimination and cross-validated RFE with supervised estimators. 


## **1. Filter Selection Methods**

### **1.1. Variance Threshold**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_01.jpg?v=1783852902" width="250">



>* Low-variance features usually add little information.
>* Removing them simplifies data before modeling.

>* Near-constant fields add little useful information.
>* Removing them simplifies high-dimensional preprocessing.

>* Low variance can still matter greatly.
>* Use domain judgment; only a first-pass filter.



In [ ]:
#@title Python Code - Variance Threshold

# Variance threshold removes nearly constant features.
# This example uses a tiny tabular dataset.
# We compute variance without machine learning.

import numpy as np
import pandas as pd

# Build a small beginner friendly dataset.
df = pd.DataFrame({
    "age_years": [22, 25, 28, 35, 40, 45],
    "country_code": [1, 1, 1, 1, 1, 1],
    "height_inches": [64, 66, 65, 70, 68, 69],
    "newsletter_flag": [0, 0, 0, 0, 1, 0],
})

# Compute variance for each numeric feature.
variances = df.var(ddof=0)
threshold = 0.05

# Keep only features above threshold.
selected = variances[variances > threshold].index.tolist()
removed = variances[variances <= threshold].index.tolist()

# Show compact results for learners.
print("Feature variances:")
for name, value in variances.items():
    print(f"{name}: {value:.3f}")

# Summarize which features remain.
print(f"Threshold: {threshold}")
print(f"Kept: {selected}")
print(f"Removed: {removed}")



### **1.2. Statistical Weakness**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_02.jpg?v=1783852904" width="250">



>* Weak features barely relate to outcomes.
>* Filters remove small, unstable, random signals.

>* Score features individually for classification or regression.
>* Remove low scorers to reduce noise.

>* Weak alone can help through interactions.
>* Filters remove irrelevant features for cleaner models.



In [ ]:
#@title Python Code - Statistical Weakness

# This example shows statistically weak features.
# We compare useful and noisy columns.
# Results support filter selection intuition.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a tiny classification dataset.
size = 120
study_hours = rng.normal(5, 1.2, size)
attendance = rng.normal(85, 8, size)
entry_day = rng.integers(1, 32, size)

# Build class probabilities from useful features.
score = 0.9 * study_hours + 0.06 * attendance
passed = (score > np.median(score)).astype(int)

data = pd.DataFrame({
    "study_hours": study_hours,
    "attendance": attendance,
    "entry_day": entry_day,
    "passed": passed,
})

# Score each feature using class mean differences.
def weakness_score(feature, target):
    group0 = feature[target == 0]
    group1 = feature[target == 1]
    pooled = feature.std(ddof=1) + 1e-9
    return abs(group1.mean() - group0.mean()) / pooled

# Compute simple univariate scores.
features = ["study_hours", "attendance", "entry_day"]
scores = {
    name: weakness_score(data[name], data["passed"])
    for name in features
}

# Sort features from strongest to weakest.
ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)

# Print a short beginner-friendly summary.
print("Univariate scores for predicting pass/fail:")
for name, value in ranked:
    print(f"{name}: {value:.3f}")

# Flag the weakest feature clearly.
weakest_name, weakest_value = ranked[-1]
print(f"Weakest feature: {weakest_name} ({weakest_value:.3f})")
print("Low scores suggest little standalone signal.")



### **1.3. Pruning Weak Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_01_03.jpg?v=1783852906" width="250">



>* Remove features with little useful signal.
>* This improves focus, efficiency, and reliability.

>* Check variation and target relationship first.
>* Remove clutter, keep genuinely informative features.

>* Pruning weak features improves generalization and trust.
>* It boosts efficiency, clarity, and robustness.



In [ ]:
#@title Python Code - Pruning Weak Features

# This example prunes weak input features.
# It uses simple variance and correlation.
# Results stay short and beginner friendly.

import numpy as np
import pandas as pd

# Set a deterministic random seed.
rng = np.random.default_rng(7)

# Create a tiny teaching dataset.
rows = 12
x_signal = np.arange(rows)
noise = rng.normal(0, 1, rows)

# Build features with different strengths.
df = pd.DataFrame({
    "study_hours": x_signal,
    "consent_signed": np.ones(rows),
    "student_id": np.arange(100, 112),
    "random_noise": noise,
})

# Create a target linked to study hours.
df["exam_score"] = 50 + 4 * x_signal

# Check dataset size safely.
assert df.shape[0] == rows
assert df.shape[1] == 5

# Measure feature variance only.
feature_names = [
    "study_hours", "consent_signed", "student_id", "random_noise"
]
variances = df[feature_names].var(ddof=0)

# Remove features with tiny variance.
variance_kept = variances[variances > 0.01].index.tolist()
variance_dropped = [
    name for name in feature_names if name not in variance_kept
]

# Score remaining features by correlation.
corrs = df[variance_kept + ["exam_score"]].corr(numeric_only=True)
strength = corrs["exam_score"].drop("exam_score").abs()

# Keep only statistically stronger features.
final_kept = strength[strength >= 0.30].index.tolist()
final_dropped = [
    name for name in variance_kept if name not in final_kept
]

# Print a compact teaching summary.
print("Original features:", feature_names)
print("Dropped for low variance:", variance_dropped)
print("Kept after variance check:", variance_kept)
print("Correlation strengths:")
print(strength.round(3).to_dict())
print("Dropped as statistically weak:", final_dropped)
print("Final pruned features:", final_kept)



## **2. Univariate Scoring Methods**

### **2.1. Classification Scoring Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_01.jpg?v=1783852897" width="250">



>* Scores each feature against class labels.
>* Fast, interpretable screening for useful predictors.

>* Choose scores based on feature type.
>* Chi-square, ANOVA, and mutual information differ.

>* Use univariate scores as an initial filter.
>* Check assumptions, interactions, and class imbalance.



### **2.2. Regression Scoring Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_02.jpg?v=1783852898" width="250">



>* Scores rank features for continuous targets.
>* Methods capture linear or broader relationships.

>* Match scoring to the relationship type.
>* Use domain judgment beyond linear patterns.

>* Univariate scores can miss combined feature value.
>* Use scoring first, then validate contextually.



### **2.3. Regression Scoring Functions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_02_03.jpg?v=1783852900" width="250">



>* Scores relate single features to numeric targets.
>* Useful screening tool, not final importance proof.

>* Regression scores detect different relationship types.
>* Choose methods matching data shape and assumptions.

>* Treat scores as clues, not proof.
>* Combine scoring with context and modeling.



## **3. Recursive Feature Elimination**

### **3.1. RFE Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_01.jpg?v=1783852891" width="250">



>* Repeatedly removes least important model features.
>* Ranks features by predictive value together.

>* Simplifies models while preserving predictive performance.
>* Keeps features that matter most together.

>* RFE needs careful stopping and costs more.
>* Useful for compact models in feature-rich domains.



### **3.2. Estimator Guided Elimination**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_02.jpg?v=1783852893" width="250">



>* Estimator ranks features by predictive usefulness.
>* Retraining repeats, capturing feature interactions and redundancy.

>* Estimators rank feature importance differently.
>* Selected features reflect model and data.

>* Selection matches prediction but depends on estimator.
>* Correlated features can mislead rankings and removal.



### **3.3. Cross Validated RFE**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_20/Lecture_A/image_03_03.jpg?v=1783852895" width="250">



>* Tests feature subsets across validation splits.
>* Finds smaller sets without hurting accuracy.

>* Cross validation tests feature subsets generalize.
>* Useful for many or correlated predictors.

>* Improves interpretation with efficient, evidence based feature reduction.
>* Results depend on model, scoring, validation, data.



# <font color="#418FDE" size="6.5" uppercase>**Filter Wrapper Methods**</font>


In this lecture, you learned to:
- Apply filter methods to remove low-variance or statistically weak features. 
- Use univariate feature selection with scoring functions appropriate to classification or regression. 
- Perform recursive feature elimination and cross-validated RFE with supervised estimators. 

In the next Lecture (Lecture B), we will go over 'Embedded Sequential'